GDAL End to End · Lab
One notebook, 15 worked examples. It follows the GDAL End to End slides: every command runs for real on the four Sentinel-2 scenes in optical_data.zip (agriculture, desert, urban, water) plus one scene read straight from the cloud.

Each example shows the same three things as the slides: what goes in, the command, and what comes out.

| # | Example | Main command | |---|---|---| | 1 | Inspect a file before touching it | gdalinfo | | 2 | Cut a window by pixels and by coordinates | gdal_translate -srcwin / -projwin | | 3 | Change format and pick bands | gdal_translate -of / -b | | 4 | Stack bands without copying pixels | gdalbuildvrt -separate | | 5 | Stretch and composites for display | gdal_translate -scale | | 6 | Band maths and the integer trap | gdal_calc.py | | 7 | Classify and threshold | gdal_calc.py | | 8 | Reproject and resample | gdalwarp -t_srs -tr -r | | 9 | Clip with a polygon | gdalwarp -cutline | | 10 | Mosaic tiles onto one grid | gdalbuildvrt · gdalwarp | | 11 | Read a cloud file without downloading | /vsicurl/ + COG | | 12 | Overviews and training chips | gdaladdo · gdal_retile.py | | 13 | Vector → raster | gdal_rasterize | | 14 | Raster → vector, cleaned | gdal_sieve.py · gdal_polygonize.py · gdal_contour | | 15 | Pan-sharpening | gdal_pansharpen.py |

At the end: delivery (COG, compression, NoData), georeferencing, and a cheat sheet. How to run: Runtime → Run all, about 4 minutes.

Step 0 · Get ready
The helper run() prints each command exactly as you would type it in a terminal, then runs it.
read() loads a result into NumPy so we can look at it.


In [ ]:
import os, sys, json, shlex, shutil, subprocess, time, zipfile
import numpy as np
import matplotlib.pyplot as plt
from osgeo import gdal, ogr, osr
gdal.UseExceptions()

if shutil.which("gdalinfo") is None:                        # Colab: add the command-line tools
    subprocess.run("apt-get -qq update && apt-get -qq install -y gdal-bin", shell=True, check=False)

def run(cmd, lines=20):
    """Print a GDAL command, run it, show the first lines of output."""
    print("$", cmd)
    args = shlex.split(cmd)
    if args[0].endswith(".py") and shutil.which(args[0]) is None:      # gdal_calc.py & friends
        args = [sys.executable, "-m", "osgeo_utils." + args[0][:-3]] + args[1:]
    r = subprocess.run(args, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().splitlines()
    if out and lines:
        print("\n".join(out[:lines]) + ("\n…" if len(out) > lines else ""))
    if r.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    print()
    return r.stdout

def read(path, band=1):
    ds = gdal.Open(path)                                    # keep the dataset alive while reading
    a = ds.GetRasterBand(band).ReadAsArray()
    ds = None
    return a

def info(path):
    return json.loads(run(f"gdalinfo -json {path}", lines=0))

def pct(a, lo=2, hi=98):
    p1, p2 = np.nanpercentile(a, [lo, hi])
    return np.clip((a - p1) / (p2 - p1 + 1e-9), 0, 1)

def rgb(paths):
    return np.dstack([pct(read(p).astype(float)) for p in paths])

def kb(path):
    return round(os.path.getsize(path) / 1024)

def panels(items, size=(5.2, 5.0), axis_off=True):
    """items: list of (draw function, title)."""
    fig, axes = plt.subplots(1, len(items), figsize=(size[0] * len(items), size[1]))
    for ax, (fn, title) in zip(np.atleast_1d(axes), items):
        fn(ax); ax.set_title(title)
        if axis_off: ax.axis("off")
    plt.tight_layout(); plt.show()

ACCENT = "#00A2FF"

# ---------- Helper: draw the mission map ("you are here") ----------
import math, textwrap
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

INK = "#0B2545"

def show_flow(steps, current=None, per_row=6):
    n = len(steps)
    cols = min(n, per_row)
    rows = math.ceil(n / cols)
    fig, ax = plt.subplots(figsize=(2.1 * cols, 1.3 * rows + 0.3))
    ax.set_xlim(0, cols)
    ax.set_ylim(rows, 0)
    ax.axis("off")
    centers = []
    for i, label in enumerate(steps):
        r, c = divmod(i, cols)
        if r % 2 == 1:                       # snake: odd rows go right-to-left
            c = cols - 1 - c
        centers.append((c + 0.5, r + 0.5))
        now = (i + 1) == current
        done = current is not None and (i + 1) < current
        face = ACCENT if now else ("#DCEBFA" if done else "white")
        ax.add_patch(FancyBboxPatch((c + 0.08, r + 0.18), 0.84, 0.64,
                                    boxstyle="round,pad=0.02,rounding_size=0.12",
                                    fc=face, ec=INK, lw=2.8 if now else 1.2))
        ax.text(c + 0.5, r + 0.5, f"{i + 1}\n" + textwrap.fill(label, 15),
                ha="center", va="center", fontsize=8.5,
                color="white" if now else INK, fontweight="bold" if now else "normal")
    for (x0, y0), (x1, y1) in zip(centers, centers[1:]):
        if y0 == y1:
            d = 0.42 if x1 > x0 else -0.42
            ax.annotate("", xy=(x1 - d, y1), xytext=(x0 + d, y0),
                        arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.3))
        else:
            ax.annotate("", xy=(x1, y1 - 0.32), xytext=(x0, y0 + 0.32),
                        arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.3))
    if current:
        ax.set_title(f"You are here: step {current} of {n}", color=INK, fontsize=11, loc="left")
    plt.tight_layout()
    plt.show()

STEPS = ['Inspect', 'Subset', 'Formats & bands', 'VRT stack', 'Stretch', 'Band maths', 'Classify', 'Reproject', 'Clip', 'Mosaic', 'Cloud COG', 'Overviews & chips', 'Rasterize', 'Polygonize', 'Pan-sharpen']
show_flow(STEPS)


# Data: unzip optical_data.zip, then copy the bands we need to short names
if not os.path.isdir("optical_data/agriculture"):
    if not os.path.exists("optical_data.zip"):
        try:
            from google.colab import files
            print("Choose optical_data.zip (it is in the image_processing_lab folder)")
            files.upload()
        except ImportError:
            raise FileNotFoundError("Put optical_data.zip next to this notebook")
    with zipfile.ZipFile("optical_data.zip") as z:
        z.extractall("optical_data", members=[m for m in z.namelist() if not m.startswith("__MACOSX")])

os.makedirs("work", exist_ok=True)
for scene in ["agriculture", "desert", "urban", "water"]:
    for b in ["B02", "B03", "B04", "B08", "B11", "B12"]:
        shutil.copy(f"optical_data/{scene}/2024-05-17-00:00_2024-05-17-23:59_Sentinel-2_L2A_{b}_(Raw).tiff",
                    f"work/{scene}_{b}.tif")
    shutil.copy(f"optical_data/{scene}/Sentinel2_RGBNIR_stack.tif", f"work/{scene}_stack.tif")
os.chdir("work")
print(len(os.listdir(".")), "files ready in the work folder")
Words you will meet today
| Word | Meaning | |---|---| | Dataset | one file as GDAL sees it: bands + geotransform + CRS + metadata | | Band | one layer of values, e.g. red or NIR | | Geotransform | 6 numbers: origin x, pixel width, 0, origin y, 0, −pixel height | | CRS / EPSG | the coordinate system, e.g. EPSG:4326 (degrees) or EPSG:32640 (UTM metres) | | NoData | the value that means "no measurement here" | | Resampling | how new pixel values are made when the grid changes (near, bilinear, average, cubic) | | Block / tile | the chunk GDAL reads at once, e.g. 256 × 256 | | Overview | a smaller copy stored inside the file for fast preview | | VRT | a small XML file that describes a dataset built from others; no pixels copied | | COG | Cloud-Optimised GeoTIFF: tiled + overviews, made for reading over HTTP | | Driver | the piece of GDAL that reads or writes one format (GTiff, PNG, GPKG …) | | /vsicurl/, /vsizip/, /vsimem/ | virtual file systems: read from the web, inside a zip, or in memory |

Example 1 · Inspect before you touch
gdalinfo answers: how big, what type, which CRS, where on Earth, what values.
The geotransform turns column and row into map coordinates: x = GT0 + col × GT1, y = GT3 + row × GT5.
Use -json whenever a script reads the output; -stats adds min, max and mean.
| | | |---|---| | Input | water_B04.tif, one Sentinel-2 red band | | Output | printed metadata + one pixel located on the map |

Commands:

gdalinfo -stats water_B04.tif
gdalinfo -json water_B04.tif


In [ ]:
show_flow(STEPS, current=1)


run("gdalinfo water_B04.tif", lines=35)
i = json.loads(run("gdalinfo -json -stats water_B04.tif", lines=0))
gt, (w, h), band = i["geoTransform"], i["size"], i["bands"][0]
print("size:", w, "×", h, "| type:", band["type"], "| CRS:", i["coordinateSystem"]["wkt"].split('"')[1])
print("geotransform:", gt)
print("min / max / mean:", band["minimum"], band["maximum"], round(band["mean"], 1))

col, row = 400, 300
x, y = gt[0] + col * gt[1], gt[3] + row * gt[5]
print(f"\npixel (col {col}, row {row}) → lon {x:.5f}, lat {y:.5f}")

a = read("water_B04.tif")
ext = [gt[0], gt[0] + gt[1] * w, gt[3] + gt[5] * h, gt[3]]
small = gdal.Open("water_B04.tif").ReadAsArray(col, row, 8, 8)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].imshow(a, cmap="gray", extent=ext); axes[0].plot(x, y, "r+", ms=20, mew=3)
axes[0].set_xlabel("Longitude (°E)"); axes[0].set_ylabel("Latitude (°N)"); axes[0].set_title("Placed on the map by the geotransform")
axes[1].imshow(small, cmap="gray")
for r_ in range(8):
    for c_ in range(8):
        axes[1].text(c_, r_, small[r_, c_], ha="center", va="center", fontsize=10,
                     color="white" if small[r_, c_] < 12 else "black")
axes[1].set_title(f"The 8 × 8 pixels at column {col}, row {row}"); axes[1].set_xlabel("column"); axes[1].set_ylabel("row")
plt.tight_layout(); plt.show()

for aux in [f for f in os.listdir(".") if f.endswith(".aux.xml")]:
    os.remove(aux)          # -stats leaves a sidecar file
Learning: a positive GT5 means the raster is upside down, and a missing CRS means the file is a picture, not a map. One second to check here saves hours later.

Example 2 · Cut a window: by pixels and by coordinates
-srcwin col row width height cuts by pixel position.
-projwin ulx uly lrx lry cuts by map coordinates, in the file's own CRS.
Cutting first makes every later step faster.
| | | |---|---| | Input | water_B04.tif (848 × 700 px) | | Output | two small GeoTIFFs, each a few hundred KB smaller |

Commands:

gdal_translate -srcwin 300 200 300 250 water_B04.tif sub_pixels.tif
gdal_translate -projwin 54.60 24.59 54.63 24.56 water_B04.tif sub_coords.tif


In [ ]:
show_flow(STEPS, current=2)


run("gdal_translate -srcwin 300 200 300 250 water_B04.tif sub_pixels.tif")
run("gdal_translate -projwin 54.60 24.59 54.63 24.56 water_B04.tif sub_coords.tif")
print(f"input {kb('water_B04.tif')} KB → window {kb('sub_pixels.tif')} KB\n")

def full(ax):
    ax.imshow(read("water_B04.tif"), cmap="gray")
    ax.add_patch(plt.Rectangle((300, 200), 300, 250, fill=False, ec="red", lw=2.5))

panels([(full, "Input with the window marked"),
        (lambda ax: ax.imshow(read("sub_pixels.tif"), cmap="gray"), "-srcwin (pixels)"),
        (lambda ax: ax.imshow(read("sub_coords.tif"), cmap="gray"), "-projwin (degrees)")])
Learning: -projwin takes upper-left then lower-right, and in EPSG:4326 the y values go down (24.59 then 24.56). Mixing them up gives an empty file.

Example 3 · Change format and pick bands
-of writes another format: PNG for a slide, GeoTIFF for work, COG for the web.
-b picks and reorders bands: the same pixels tell a different story.
-scale is needed for PNG because PNG stores 8-bit values only.
| | | |---|---| | Input | water_B04.tif, agriculture_stack.tif (4 bands: R, G, B, NIR) | | Output | a PNG picture and two 3-band composites |

Commands:

gdal_translate -of PNG -scale water_B04.tif water_B04.png
gdal_translate -b 1 -b 2 -b 3 agriculture_stack.tif agri_true.tif    # true colour
gdal_translate -b 4 -b 1 -b 2 agriculture_stack.tif agri_cir.tif     # NIR, red, green


In [ ]:
show_flow(STEPS, current=3)


run("gdal_translate -of PNG -scale water_B04.tif water_B04.png")
run("gdal_translate -b 1 -b 2 -b 3 agriculture_stack.tif agri_true.tif")
run("gdal_translate -b 4 -b 1 -b 2 agriculture_stack.tif agri_cir.tif")
print(f"GeoTIFF {kb('water_B04.tif')} KB → PNG {kb('water_B04.png')} KB (PNG keeps no CRS of its own)\n")

def stack(path):
    return lambda ax: ax.imshow(gdal.Open(path).ReadAsArray().transpose(1, 2, 0))

panels([(lambda ax: ax.imshow(plt.imread("water_B04.png"), cmap="gray"), "-of PNG -scale"),
        (stack("agri_true.tif"), "-b 1 -b 2 -b 3: true colour"),
        (stack("agri_cir.tif"), "-b 4 -b 1 -b 2: vegetation red")])
Learning: band order is a decision, not a detail. Colour infrared (NIR, red, green) makes healthy fields jump out in red, which is why Ahmad's mission uses it.

Example 4 · Stack bands without copying pixels (VRT)
gdalbuildvrt -separate writes a small XML file that points to the band files.
Without -separate, GDAL mosaics the files side by side instead of stacking them.
Compare it with gdal_merge.py, which copies every pixel into a new file.
| | | |---|---| | Input | three single-band files (B04, B03, B02) | | Output | agri_rgb.vrt, a text file of about 2 KB, that behaves like a 3-band image |

Commands:

gdalbuildvrt -separate agri_rgb.vrt agriculture_B04.tif agriculture_B03.tif agriculture_B02.tif
gdal_merge.py -separate -o agri_rgb_copy.tif agriculture_B04.tif agriculture_B03.tif agriculture_B02.tif


In [ ]:
show_flow(STEPS, current=4)


run("gdalbuildvrt -separate agri_rgb.vrt agriculture_B04.tif agriculture_B03.tif agriculture_B02.tif")
run("gdal_merge.py -separate -o agri_rgb_copy.tif agriculture_B04.tif agriculture_B03.tif agriculture_B02.tif", lines=3)
print(f"VRT: {os.path.getsize('agri_rgb.vrt')} bytes   ·   real copy: {kb('agri_rgb_copy.tif'):,} KB\n")
print(open("agri_rgb.vrt").read()[:600], "…\n")

def tint(name, ch):
    def f(ax):
        a = pct(read(f"agriculture_{name}.tif").astype(float))
        img = np.zeros(a.shape + (3,)); img[:, :, ch] = a
        ax.imshow(img)
    return f

panels([(tint("B04", 0), "B04 red"), (tint("B03", 1), "B03 green"), (tint("B02", 2), "B02 blue"),
        (lambda ax: ax.imshow(rgb(["agriculture_B04.tif", "agriculture_B03.tif", "agriculture_B02.tif"])),
         "The VRT, read as one image")], size=(4.6, 4.6))
Learning: a VRT is a recipe, not a copy. Move or rename the band files and the VRT breaks, so keep relative paths and ship the VRT with its data.

Example 5 · Stretch and composites (display only)
Satellite values sit in a narrow range, so a raw composite looks dark.
-scale in_min in_max 0 255 spreads them; -exponent applies gamma.
Different band triplets answer different questions.
| | | |---|---| | Input | agri_rgb.vrt + the urban SWIR bands | | Output | 8-bit pictures for slides and reports |

Commands:

gdal_translate -ot Byte -scale_1 9 127 0 255 -scale_2 15 108 0 255 -scale_3 2 87 0 255 agri_rgb.vrt agri_stretch.tif
gdalbuildvrt -separate urban_swir.vrt urban_B12.tif urban_B11.tif urban_B04.tif


In [ ]:
show_flow(STEPS, current=5)


tc = gdal.Open("agri_rgb.vrt").ReadAsArray()
lo, hi = np.percentile(tc, [2, 98], axis=(1, 2)).astype(int)
scale = " ".join(f"-scale_{k+1} {lo[k]} {hi[k]} 0 255" for k in range(3))
print("percentile 2–98 per band:", list(zip(lo, hi)), "\n")
run(f"gdal_translate -ot Byte {scale} agri_rgb.vrt agri_stretch.tif")
run(f"gdal_translate -ot Byte {scale} -exponent 0.7 agri_rgb.vrt agri_gamma.tif")
run("gdalbuildvrt -separate urban_swir.vrt urban_B12.tif urban_B11.tif urban_B04.tif")

def stack(path, div=255):
    return lambda ax: ax.imshow(gdal.Open(path).ReadAsArray().transpose(1, 2, 0) / div)

panels([(stack("agri_rgb.vrt"), "Raw values: dark"), (stack("agri_stretch.tif"), "-scale 2–98 %"),
        (stack("agri_gamma.tif"), "-scale + gamma 0.7"),
        (lambda ax: ax.imshow(rgb(["urban_B12.tif", "urban_B11.tif", "urban_B04.tif"])), "Urban SWIR composite")],
       size=(4.6, 4.6))
Learning: stretching changes the look, never the measurement. Always compute indices from the raw file, and keep the stretched copy for slides only.

Example 6 · Band maths, and the integer trap
NDVI = (NIR − Red) / (NIR + Red). One formula, any band pair.
These bands are 8-bit integers: subtraction below zero wraps around and the sum can overflow.
Cast to float inside --calc and the problem disappears.
| | | |---|---| | Input | agriculture_B08.tif (NIR) and agriculture_B04.tif (red) | | Output | ndvi.tif, Float32, values −1 to 1 |

Commands:

gdal_calc.py -A agriculture_B08.tif -B agriculture_B04.tif --outfile=ndvi_int.tif --calc="(A-B)/(A+B)" --type=Float32
gdal_calc.py -A agriculture_B08.tif -B agriculture_B04.tif --outfile=ndvi.tif \
  --calc="(A.astype(float)-B)/(A.astype(float)+B+1e-10)" --type=Float32


In [ ]:
show_flow(STEPS, current=6)


run('gdal_calc.py -A agriculture_B08.tif -B agriculture_B04.tif --outfile=ndvi_int.tif --calc="(A-B)/(A+B)" --type=Float32 --overwrite --quiet')
run('gdal_calc.py -A agriculture_B08.tif -B agriculture_B04.tif --outfile=ndvi.tif --calc="(A.astype(float)-B)/(A.astype(float)+B+1e-10)" --type=Float32 --overwrite --quiet')

wrong, right = read("ndvi_int.tif"), read("ndvi.tif")
bad = np.abs(np.nan_to_num(wrong, posinf=9, neginf=-9) - right) > 0.01
print(f"integer version: min {np.nanmin(wrong):.2f}, max {np.nanmax(wrong)}")
print(f"float version  : min {right.min():.2f}, max {right.max():.2f}")
print(f"{100 * bad.mean():.1f} % of pixels are wrong in the integer version")

panels([(lambda ax: ax.imshow(np.clip(wrong, -1, 1), cmap="RdYlGn", vmin=-0.6, vmax=0.8), 'Integer "(A-B)/(A+B)"'),
        (lambda ax: ax.imshow(right, cmap="RdYlGn", vmin=-0.6, vmax=0.8), "Cast to float first"),
        (lambda ax: ax.imshow(bad, cmap="Reds"), "Wrong pixels in red")])
Learning: this is the most common GDAL bug in remote sensing code, and it is silent: the map still looks plausible. Check the minimum and maximum of every index you compute.

Example 7 · Classify and threshold
The same tool turns a continuous index into classes or a yes/no mask.
Multiply conditions to build classes: 1*(A<0.2) + 2*((A>=0.2)*(A<0.4)) + …
A mask is just a threshold written as A>0.2, stored as Byte.
| | | |---|---| | Input | ndvi.tif, and green + NIR for NDWI | | Output | ndvi_class.tif (4 classes) and water_mask.tif (0/1) |

Commands:

gdal_calc.py -A ndvi.tif --outfile=ndvi_class.tif --type=Byte \
  --calc="1*(A<0.2) + 2*((A>=0.2)*(A<0.4)) + 3*((A>=0.4)*(A<0.6)) + 4*(A>=0.6)"
gdal_calc.py -A ndwi.tif --outfile=water_mask.tif --type=Byte --calc="A>0.2"


In [ ]:
show_flow(STEPS, current=7)


run('gdal_calc.py -A ndvi.tif --outfile=ndvi_class.tif --type=Byte --overwrite --quiet --calc="1*(A<0.2) + 2*((A>=0.2)*(A<0.4)) + 3*((A>=0.4)*(A<0.6)) + 4*(A>=0.6)"')
run('gdal_calc.py -A water_B03.tif -B water_B08.tif --outfile=ndwi.tif --type=Float32 --overwrite --quiet --calc="(A.astype(float)-B)/(A.astype(float)+B+1e-10)"')
run('gdal_calc.py -A ndwi.tif --outfile=water_mask.tif --type=Byte --overwrite --quiet --calc="A>0.2"')

cls = read("ndvi_class.tif")
gt = json.loads(run("gdalinfo -json ndvi.tif", lines=0))["geoTransform"]
px_m = abs(gt[1]) * 111320 * np.cos(np.radians(gt[3])), abs(gt[5]) * 110574
ha = px_m[0] * px_m[1] / 1e4
for k, name in [(1, "bare / sand"), (2, "sparse"), (3, "moderate"), (4, "dense")]:
    n = int((cls == k).sum())
    print(f"class {k} ({name}): {n:>7,} px ≈ {n * ha:>8,.0f} ha")
print(f"\nwater in the coastal scene: {100 * read('water_mask.tif').mean():.1f} % of pixels")

panels([(lambda ax: ax.imshow(read("ndvi.tif"), cmap="RdYlGn", vmin=-0.2, vmax=0.8), "NDVI"),
        (lambda ax: ax.imshow(cls, cmap="YlGn", vmin=0, vmax=4), "4 classes"),
        (lambda ax: ax.imshow(read("ndwi.tif"), cmap="BrBG", vmin=-0.6, vmax=0.6), "NDWI"),
        (lambda ax: ax.imshow(read("water_mask.tif"), cmap="Blues"), "Water mask")], size=(4.6, 4.6))
Learning: class boundaries are a choice you must defend. In the UAE, bright sand already reaches NDVI ≈ 0.15, so 0.2 is the lowest sensible line between bare ground and plants.

Example 8 · Reproject and resample
-t_srs sets the target CRS; -tr sets the pixel size in the target units.
Degrees are not metres: to measure area or distance, move to UTM first.
-r chooses how new values are made: near keeps classes, average and cubic smooth measurements.
| | | |---|---| | Input | water_B04.tif in EPSG:4326 (degrees) | | Output | water_utm.tif in EPSG:32640 at exactly 10 m, plus three 60 m versions |

Commands:

gdalwarp -t_srs EPSG:32640 -tr 10 10 -r near water_B04.tif water_utm.tif
gdalwarp -tr 60 60 -r average water_utm.tif water_60_average.tif


In [ ]:
show_flow(STEPS, current=8)


run("gdalwarp -overwrite -t_srs EPSG:32640 -tr 10 10 -r near water_B04.tif water_utm.tif")
for m in ["near", "average", "cubic"]:
    run(f"gdalwarp -overwrite -tr 60 60 -r {m} water_utm.tif water_60_{m}.tif", lines=2)

before, after = info("water_B04.tif"), info("water_utm.tif")
print("before:", before["size"], "pixels of", round(before["geoTransform"][1], 7), "degrees")
print("after :", after["size"], "pixels of", after["geoTransform"][1], "metres")

cw = (slice(20, 40), slice(10, 30))
panels([(lambda ax: ax.imshow(read("water_utm.tif"), cmap="gray"), "EPSG:32640, 10 m")] +
       [(lambda ax, m=m: ax.imshow(read(f"water_60_{m}.tif")[cw], cmap="gray", interpolation="nearest"),
         f"60 m, -r {m}") for m in ["near", "average", "cubic"]], size=(4.6, 4.6))
Learning: every warp resamples the pixels, so do it once, as late as possible. Warping a warped file loses a little detail each time.

Example 9 · Clip with a polygon
-cutline takes any vector file; GDAL reprojects the polygon for you.
Without -crop_to_cutline you keep the full extent, with the outside set to NoData.
Always set -dstnodata, otherwise the outside becomes 0 and pollutes statistics.
| | | |---|---| | Input | water_utm.tif + aoi.geojson (one polygon) | | Output | water_clip.tif, cropped to the polygon, outside = NoData |

Commands:

gdalwarp -cutline aoi.geojson -crop_to_cutline -dstnodata 0 water_utm.tif water_clip.tif


In [ ]:
show_flow(STEPS, current=9)


aoi = {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {"name": "port"}, "geometry": {
    "type": "Polygon", "coordinates": [[[54.617, 24.584], [54.635, 24.586], [54.641, 24.564],
                                        [54.623, 24.554], [54.609, 24.570], [54.617, 24.584]]]}}]}
json.dump(aoi, open("aoi.geojson", "w"))
run("gdalwarp -overwrite -cutline aoi.geojson -crop_to_cutline -dstnodata 0 water_utm.tif water_clip.tif")
print("before:", info("water_utm.tif")["size"], "→ after:", info("water_clip.tif")["size"])

poly = np.array(aoi["features"][0]["geometry"]["coordinates"][0])
gt = json.loads(run("gdalinfo -json water_B04.tif", lines=0))["geoTransform"]
a = read("water_B04.tif")
ext = [gt[0], gt[0] + gt[1] * a.shape[1], gt[3] + gt[5] * a.shape[0], gt[3]]

def before(ax):
    ax.imshow(a, cmap="gray", extent=ext); ax.plot(poly[:, 0], poly[:, 1], color="red", lw=2.5)

panels([(before, "Input + cutline"),
        (lambda ax: ax.imshow(np.ma.masked_equal(read("water_clip.tif"), 0), cmap="gray"), "-crop_to_cutline")],
       size=(6.5, 5.6))
Learning: the polygon can be in any CRS. Reproject the polygon, never the raster, when they disagree: it is instant and lossless.

Example 10 · Mosaic tiles onto one grid
gdalbuildvrt joins tiles instantly and for free, but only if they share a CRS.
gdalwarp is the one that works across projections, resolutions and grids.
Here we cut one scene into two overlapping tiles, then put it back together.
| | | |---|---| | Input | tile_west.tif and tile_east.tif (overlapping) | | Output | one seamless mosaic: a VRT, and a real GeoTIFF in UTM |

Commands:

gdalbuildvrt mosaic.vrt tile_west.tif tile_east.tif
gdalwarp -t_srs EPSG:32640 -tr 10 10 tile_west.tif tile_east.tif mosaic_utm.tif


In [ ]:
show_flow(STEPS, current=10)


run("gdal_translate -srcwin 0 0 500 700 water_B04.tif tile_west.tif", lines=2)
run("gdal_translate -srcwin 420 0 428 700 water_B04.tif tile_east.tif", lines=2)
run("gdalbuildvrt mosaic.vrt tile_west.tif tile_east.tif")
run("gdalwarp -overwrite -t_srs EPSG:32640 -tr 10 10 tile_west.tif tile_east.tif mosaic_utm.tif", lines=3)
print("west:", info("tile_west.tif")["size"], "· east:", info("tile_east.tif")["size"],
      "· mosaic:", info("mosaic.vrt")["size"], f"({os.path.getsize('mosaic.vrt')} bytes)")

panels([(lambda ax: ax.imshow(read("tile_west.tif"), cmap="gray"), "tile_west"),
        (lambda ax: ax.imshow(read("tile_east.tif"), cmap="gray"), "tile_east"),
        (lambda ax: ax.imshow(read("mosaic.vrt"), cmap="gray"), "gdalbuildvrt mosaic")])
Learning: in the overlap, the last file listed wins. Order your tiles by quality (least cloud last) and the mosaic cleans itself up. Across UTM zones, gdalbuildvrt refuses: that is Mission 1's lesson.

Example 11 · Read a cloud file without downloading it
A COG stores tiles plus smaller overview copies, so a reader can fetch just one part.
/vsicurl/ in front of a URL opens the file over HTTP.
-srcwin fetches full detail for a small area; -outsize uses the overviews for a quick look.
| | | |---|---| | Input | a 10 m Sentinel-2 band in the cloud (about 240 MB) | | Output | a 512 × 512 window and a 5 % preview, each a fraction of a MB |

Commands:

gdalinfo /vsicurl/https://…/S2B_40RBN_20260316_0_L2A/B04.tif
gdal_translate -srcwin 5200 4600 512 512 /vsicurl/https://…/B04.tif cloud_win.tif
gdal_translate -outsize 5% 5% /vsicurl/https://…/B04.tif cloud_ovr.tif


In [ ]:
show_flow(STEPS, current=11)


url = ("/vsicurl/https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/40/R/BN/2026/3/"
       "S2B_40RBN_20260316_0_L2A/B04.tif")
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"       # do not list the whole bucket
run(f"gdalinfo {url}", lines=28)

t = time.time(); run(f"gdal_translate -srcwin 5200 4600 512 512 {url} cloud_win.tif", lines=2); t_win = time.time() - t
t = time.time(); run(f"gdal_translate -outsize 5% 5% {url} cloud_ovr.tif", lines=2); t_ovr = time.time() - t
w, h = info(url)["size"]
print(f"full band ≈ {w * h * 2 / 1e6:.0f} MB")
print(f"window   : {kb('cloud_win.tif')} KB in {t_win:.1f} s")
print(f"preview  : {kb('cloud_ovr.tif')} KB in {t_ovr:.1f} s")

def ovr(ax):
    a = pct(read("cloud_ovr.tif").astype(float)); s = a.shape[1] / w
    ax.imshow(a, cmap="gray")
    ax.add_patch(plt.Rectangle((5200 * s, 4600 * s), 512 * s, 512 * s, fill=False, ec="red", lw=2))

panels([(ovr, "-outsize 5 % (reads overviews)"),
        (lambda ax: ax.imshow(pct(read("cloud_win.tif").astype(float)), cmap="gray"), "-srcwin (full detail)")],
       size=(6.5, 6))
Learning: this is how every modern map viewer works. If your own files are COGs, your users get the same speed with no server code at all.

Example 12 · Overviews and training chips
gdaladdo adds smaller copies inside the file: previews stop re-reading every pixel.
Overviews make the file a little larger; that is the trade for speed.
gdal_retile.py cuts a scene into fixed-size chips, the usual first step for deep learning.
| | | |---|---| | Input | water_utm.tif | | Output | the same file with 4 overview levels, and a folder of 256 × 256 chips |

Commands:

gdaladdo -r average water_ovr.tif 2 4 8 16
gdal_retile.py -ps 256 256 -targetDir chips water_utm.tif


In [ ]:
show_flow(STEPS, current=12)


shutil.copy("water_utm.tif", "water_ovr.tif")
before = kb("water_ovr.tif")
run("gdaladdo -r average water_ovr.tif 2 4 8 16")
after = kb("water_ovr.tif")
levels = [o["size"] for o in info("water_ovr.tif")["bands"][0].get("overviews", [])]
print(f"{before:,} KB → {after:,} KB  (+{100 * (after - before) / before:.0f} %) for {len(levels)} overviews: {levels}\n")

shutil.rmtree("chips", ignore_errors=True); os.makedirs("chips")
run("gdal_retile.py -ps 256 256 -targetDir chips water_utm.tif", lines=2)
chips = sorted(f for f in os.listdir("chips") if f.endswith(".tif"))
print(f"{len(chips)} chips written, first three: {chips[:3]}")

fig, axes = plt.subplots(2, max(4, len(chips) // 2 + 1), figsize=(16, 6))
for ax in axes.ravel(): ax.axis("off")
for ax, c in zip(axes.ravel(), chips):
    ax.imshow(read(f"chips/{c}"), cmap="gray"); ax.set_title(c.split("_")[-1], fontsize=9)
plt.suptitle("gdal_retile.py output: ready-made training chips"); plt.tight_layout(); plt.show()
Learning: for training data, add -overlap so objects on a chip border also appear whole in a neighbour, and split train/test by area, not by chip.

Example 13 · Vector → raster
gdal_rasterize burns polygons, lines or points into a pixel grid.
-burn writes a fixed value; -a field writes the value of an attribute.
Match the target grid with -ts (size) and -te (extent) or the two layers will not line up.
| | | |---|---| | Input | aoi.geojson (1 polygon) + the grid of water_mask.tif | | Output | aoi_mask.tif: 1 inside the polygon, 0 outside |

Commands:

gdal_rasterize -burn 1 -ot Byte -ts 848 700 -te 54.5746 24.5415 54.6508 24.5986 aoi.geojson aoi_mask.tif


In [ ]:
show_flow(STEPS, current=13)


i = info("water_mask.tif"); gt, (w, h) = i["geoTransform"], i["size"]
te = f"{gt[0]} {gt[3] + h * gt[5]} {gt[0] + w * gt[1]} {gt[3]}"        # xmin ymin xmax ymax
run(f"gdal_rasterize -burn 1 -ot Byte -ts {w} {h} -te {te} aoi.geojson aoi_mask.tif")

mask = read("aoi_mask.tif")
px_m = abs(gt[1]) * 111320 * np.cos(np.radians(gt[3]))
print(f"{mask.sum():,} pixels inside ≈ {mask.sum() * px_m ** 2 / 1e6:.1f} km²")

poly = np.array(json.load(open("aoi.geojson"))["features"][0]["geometry"]["coordinates"][0])
def vec(ax):
    ax.fill(poly[:, 0], poly[:, 1], color="#DCEBFA"); ax.plot(poly[:, 0], poly[:, 1], color="#0B2545", lw=2.5)
    ax.set_aspect("equal")

panels([(vec, "Input: one polygon"), (lambda ax: ax.imshow(mask, cmap="gray"), "Output: pixel mask"),
        (lambda ax: ax.imshow(np.ma.masked_where(mask == 0, read("water_B04.tif")), cmap="gray"), "Mask applied")])
Learning: this is how Aisha turns OpenStreetMap roads into a mask, so the car detector only looks at road pixels. A mask is the cheapest way to cut a problem down.

Example 14 · Raster → vector, cleaned
gdal_sieve.py deletes patches smaller than -st pixels: the noise goes first.
gdal_polygonize.py turns each remaining patch into a polygon with its value as an attribute.
gdal_contour draws lines at chosen levels of a continuous surface.
| | | |---|---| | Input | water_mask.tif (0/1) and ndwi.tif (continuous) | | Output | GeoPackages: cleaned water polygons and a shoreline |

Commands:

gdal_sieve.py -st 50 -4 water_mask.tif water_sieved.tif
gdal_polygonize.py water_sieved.tif -f GPKG water_clean.gpkg water value
gdal_contour -a ndwi -fl 0.2 ndwi_smooth.tif shore.gpkg


In [ ]:
show_flow(STEPS, current=14)


run("gdal_sieve.py -st 50 -4 water_mask.tif water_sieved.tif", lines=2)
for f in ["water_raw.gpkg", "water_clean.gpkg", "shore.gpkg"]:
    if os.path.exists(f): os.remove(f)
run("gdal_polygonize.py water_mask.tif -f GPKG water_raw.gpkg water value", lines=2)
run("gdal_polygonize.py water_sieved.tif -f GPKG water_clean.gpkg water value", lines=2)
run("gdalwarp -overwrite -tr 0.0006 0.0006 -r average ndwi.tif ndwi_smooth.tif", lines=2)
run("gdal_contour -a ndwi -fl 0.2 ndwi_smooth.tif shore.gpkg", lines=2)
run("ogrinfo -so water_clean.gpkg water", lines=12)

def count(path):
    ds = ogr.Open(path); n = sum(1 for f in ds.GetLayer() if f.GetField("value") == 1); ds = None
    return n
print(f"water polygons: {count('water_raw.gpkg')} → {count('water_clean.gpkg')} after the sieve")

ds = ogr.Open("shore.gpkg"); lines = [np.array(f.GetGeometryRef().GetPoints()) for f in ds.GetLayer()]; ds = None
gt = json.loads(run("gdalinfo -json water_B04.tif", lines=0))["geoTransform"]
a = read("water_B04.tif"); ext = [gt[0], gt[0] + gt[1] * a.shape[1], gt[3] + gt[5] * a.shape[0], gt[3]]

def shore(ax):
    ax.imshow(rgb(["water_B04.tif", "water_B03.tif", "water_B02.tif"]), extent=ext)
    for p in lines:
        if len(p) > 10: ax.plot(p[:, 0], p[:, 1], color="yellow", lw=1.4)

panels([(lambda ax: ax.imshow(read("water_mask.tif"), cmap="gray"), "Mask: noisy"),
        (lambda ax: ax.imshow(read("water_sieved.tif"), cmap="gray"), "After gdal_sieve -st 50"),
        (shore, "gdal_contour: shoreline")])
Learning: sieve before you polygonize. Otherwise you create thousands of single-pixel polygons and every later step is slow.

Example 15 · Pan-sharpening
Some satellites carry a sharp grey (panchromatic) band plus coarser colour bands.
gdal_pansharpen.py puts the detail of the grey band into the colour bands.
Sentinel-2 has no pan band, so we simulate one from the 10 m bands to see the effect.
| | | |---|---| | Input | a sharp grey band + three 4× coarser colour bands | | Output | sharp.tif: colour at the grey band's detail |

Commands:

gdal_calc.py -A agriculture_B04.tif -B agriculture_B03.tif -C agriculture_B02.tif \
  --outfile=pan.tif --calc="(A.astype(float)+B+C)/3" --type=Byte
gdal_pansharpen.py -r cubic pan.tif coarse_B04.tif coarse_B03.tif coarse_B02.tif sharp.tif


In [ ]:
show_flow(STEPS, current=15)


for b in ["B04", "B03", "B02"]:
    run(f"gdalwarp -overwrite -tr 0.00036 0.00032 -r average agriculture_{b}.tif coarse_{b}.tif", lines=2)
run('gdal_calc.py -A agriculture_B04.tif -B agriculture_B03.tif -C agriculture_B02.tif --outfile=pan.tif --calc="(A.astype(float)+B+C)/3" --type=Byte --overwrite --quiet')
run("gdal_pansharpen.py -r cubic pan.tif coarse_B04.tif coarse_B03.tif coarse_B02.tif sharp.tif", lines=3)
print("pan:", info("pan.tif")["size"], "· coarse colour:", info("coarse_B04.tif")["size"],
      "· pan-sharpened:", info("sharp.tif")["size"])

win = (slice(250, 450), slice(350, 600)); cw = (slice(62, 112), slice(87, 150))
def sharp(ax):
    a = np.dstack([pct(read("sharp.tif", k).astype(float)) for k in (1, 2, 3)])
    ax.imshow(a[win])

panels([(lambda ax: ax.imshow(read("pan.tif")[win], cmap="gray"), "Simulated pan band (sharp)"),
        (lambda ax: ax.imshow(rgb(["coarse_B04.tif", "coarse_B03.tif", "coarse_B02.tif"])[cw], interpolation="nearest"),
         "Colour, 4× coarser"),
        (sharp, "gdal_pansharpen output"),
        (lambda ax: ax.imshow(rgb(["agriculture_B04.tif", "agriculture_B03.tif", "agriculture_B02.tif"])[win]),
         "Original 10 m colour")], size=(4.6, 4.6))
Learning: pan-sharpening invents plausible detail; it does not recover information. Use it for display, and measure on the original bands. Our simulated pan also shifts the colours slightly, because a real pan band would be matched to the colour bands.

Finishing the job · deliver files others can open
COG for anything shared: tiles + overviews, readable over HTTP.
Compression is free to test. PREDICTOR=2 for integers, 3 for floats, but always measure: it does not always win.
NoData must be declared, or the empty border is counted in every statistic.


In [ ]:
sizes = {}
for k, (name, opt) in enumerate({"none": "-co COMPRESS=NONE", "LZW": "-co COMPRESS=LZW",
                                 "DEFLATE": "-co COMPRESS=DEFLATE",
                                 "DEFLATE+PREDICTOR": "-co COMPRESS=DEFLATE -co PREDICTOR=3",
                                 "COG": "-of COG -co COMPRESS=DEFLATE"}.items()):
    run(f"gdal_translate {opt} ndvi.tif ndvi_c{k}.tif", lines=0)
    sizes[name] = kb(f"ndvi_c{k}.tif")
print("Float NDVI, KB:", sizes)

int_sizes = {}
for k, (name, opt) in enumerate({"none": "-co COMPRESS=NONE", "LZW": "-co COMPRESS=LZW",
                                 "DEFLATE": "-co COMPRESS=DEFLATE",
                                 "DEFLATE+PREDICTOR": "-co COMPRESS=DEFLATE -co PREDICTOR=2"}.items()):
    run(f"gdal_translate {opt} water_B04.tif b04_c{k}.tif", lines=0)
    int_sizes[name] = kb(f"b04_c{k}.tif")
print("8-bit band, KB:", int_sizes)

run("gdal_translate water_clip.tif clip_nd.tif -a_nodata none", lines=0)
mean_no = json.loads(run("gdalinfo -json -stats clip_nd.tif", lines=0))["bands"][0]["mean"]
run("gdal_edit.py -a_nodata 0 clip_nd.tif", lines=0)
mean_yes = json.loads(run("gdalinfo -json -stats clip_nd.tif", lines=0))["bands"][0]["mean"]
print(f"\nmean without NoData: {mean_no:.1f}   ·   with NoData declared: {mean_yes:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (data, title) in zip(axes, [(sizes, "Float NDVI"), (int_sizes, "8-bit band")]):
    names = list(data)[::-1]; vals = [data[n] for n in names]
    ax.barh(names, vals, color=[ACCENT if v == min(vals) else "#9DB4CC" for v in vals])
    ax.set_xlabel("KB"); ax.set_title(title)
plt.tight_layout(); plt.show()
Georeferencing: giving an image its place
Assign (-a_srs, -a_ullr): you already know the corner coordinates. Metadata only, instant, no pixel touched.
Transform (gdalwarp -t_srs): move pixels onto a new grid. Every pixel is recomputed.
GCPs (-gcp col row x y): for a scan or a photo, mark known points and let gdalwarp fit the image to the map.


run("gdal_translate -of PNG -co WORLDFILE=NO urban_B04.tif urban_plain.png", lines=2)
for f in [f for f in os.listdir(".") if f.startswith("urban_plain.png.aux")]:
    os.remove(f)
print("CRS of the bare PNG:", json.loads(run("gdalinfo -json urban_plain.png", lines=0)).get("coordinateSystem", "none"), "\n")

i = info("urban_B04.tif"); gt, (w, h) = i["geoTransform"], i["size"]
ulx, uly = gt[0], gt[3]
lrx, lry = gt[0] + w * gt[1], gt[3] + h * gt[5]
run(f"gdal_translate -a_srs EPSG:4326 -a_ullr {ulx} {uly} {lrx} {lry} urban_plain.png urban_assigned.tif", lines=2)

gcps = " ".join(f"-gcp {c} {r} {gt[0] + c * gt[1]} {gt[3] + r * gt[5]}" for c, r in [(0, 0), (w, 0), (0, h), (w, h)])
run(f"gdal_translate -a_srs EPSG:4326 {gcps} urban_plain.png urban_gcp.tif", lines=2)
run("gdalwarp -overwrite -order 1 -t_srs EPSG:32640 urban_gcp.tif urban_gcp_utm.tif", lines=3)
print("assigned corners match the original:", np.allclose(gdal.Open("urban_assigned.tif").GetGeoTransform(), gt))
Cheat sheet
| Job | Command | |---|---| | What is in this file? | gdalinfo -stats -json file.tif | | Cut a piece | gdal_translate -srcwin / -projwin | | Change format, pick bands | gdal_translate -of -b -scale | | Stack bands, mosaic for free | gdalbuildvrt -separate / gdalbuildvrt | | Band maths, classes, masks | gdal_calc.py --calc="…" (cast to float!) | | Reproject, resample, clip, mosaic | gdalwarp -t_srs -tr -r -cutline | | Read cloud files | /vsicurl/ + COG | | Speed up viewing | gdaladdo -r average file.tif 2 4 8 16 | | Cut training chips | gdal_retile.py -ps 256 256 | | Vector → raster | gdal_rasterize -burn 1 -ts -te | | Raster → vector | gdal_sieve.py then gdal_polygonize.py | | Lines from a surface | gdal_contour -fl | | Sharpen colour with a grey band | gdal_pansharpen.py | | Deliver | gdal_translate -of COG -co COMPRESS=DEFLATE | | Fix metadata in place | gdal_edit.py -a_nodata 0 -a_srs EPSG:4326 |

Try it yourself
Run Examples 6–7 on the desert scene. What is the highest NDVI you find, and is it a plant?
In Example 8, reproject to EPSG:32639 instead of 32640. How much does the pixel grid shift?
In Example 12, add -overlap 32 to gdal_retile.py. How many chips do you get now?
Take any result and make it a COG, then open it in QGIS from the file path alone.
